# DigitSense — Handwritten Digit Recognition (0–9)

Complete, reproducible machine learning pipeline for handwritten digit classification (`sklearn.datasets.load_digits`):
1. **Dataset Validation & EDA** (`1,797` samples, `8 × 8` grayscale pixels, `0–16` intensity)
2. **Feature Engineering & Dimensionality Reduction** (`64` pixels → `82` engineered features → `StandardScaler` → `PCA` retaining 95% variance / `38` dimensions)
3. **Leak-Free Model Comparison** via 5-fold Stratified Cross-Validation (**RBF SVM**, **Random Forest**, **K-Nearest Neighbors**, **Logistic Regression**)
4. **Held-Out Test Evaluation** (`80/20` stratified split, `random_state=42`) & per-class classification report
5. **All 6 Output Visualizations** rendered inline and saved to `./outputs/`
6. **Interactive Image Upload & Inference** using `google.colab.files.upload()`

## 1. Imports & Styling Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DigitSense color palette
ACCENT_ORANGE = "#d4571f"
ACCENT_TEAL = "#1f6f6b"
INK_DARK = "#1d1b18"
MUTED_GRAY = "#6b665e"
SOFT_SAND = "#f1ece3"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#d8d2c7",
    "axes.labelcolor": INK_DARK,
    "text.color": INK_DARK,
    "xtick.color": MUTED_GRAY,
    "ytick.color": MUTED_GRAY,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})
print("Environment configured successfully.")

## 2. Dataset Loading & Validation

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

n_samples, n_features = X.shape
missing_count = int(np.isnan(X).sum())
min_val, max_val = float(X.min()), float(X.max())

print(f"Samples        : {n_samples:,}")
print(f"Raw features   : {n_features} (8x8 grayscale pixels)")
print(f"Missing values : {missing_count}")
print(f"Pixel range    : {min_val:.0f} to {max_val:.0f}")
print(f"Classes        : {len(np.unique(y))} ({list(np.unique(y))})")

assert n_samples == 1797
assert n_features == 64
assert missing_count == 0

## 3. Exploratory Data Analysis (4 Visualizations)

In [ ]:
imgs = X.reshape(-1, 8, 8)

# 1. Class Distribution
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=130)
classes, counts = np.unique(y, return_counts=True)
bars = ax.bar(classes, counts, color=ACCENT_TEAL, width=0.65)
ax.axhline(counts.mean(), color=ACCENT_ORANGE, linestyle="--", linewidth=1.4,
           label=f"Mean count ({counts.mean():.1f})")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, count + 2.5, str(count),
            ha="center", va="bottom", fontsize=9.5, fontweight="bold")
ax.set_xticks(classes)
ax.set_xlabel("Digit Class (0–9)")
ax.set_ylabel("Number of Samples")
ax.set_title("Class Distribution Across Handwritten Digits (n = 1,797)")
ax.set_ylim(0, max(counts) * 1.15)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"))
plt.show()

# 2. Sample Gallery (3 examples per digit class)
fig, axes = plt.subplots(3, 10, figsize=(11, 4.2), dpi=130)
for digit in range(10):
    digit_indices = np.where(y == digit)[0][:3]
    for row_idx, sample_idx in enumerate(digit_indices):
        ax = axes[row_idx, digit]
        ax.imshow(imgs[sample_idx], cmap="gray_r", vmin=0, vmax=16)
        ax.set_xticks([])
        ax.set_yticks([])
        if row_idx == 0:
            ax.set_title(f"Digit {digit}", fontsize=10, fontweight="bold")
fig.suptitle("Sample Gallery: Curated 8×8 Grayscale Digits (3 Examples per Class)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "sample_gallery.png"), bbox_inches="tight")
plt.show()

# 3. Mean Digit Patterns
fig, axes = plt.subplots(2, 5, figsize=(10.5, 4.8), dpi=130)
for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    mean_img = imgs[y == digit].mean(axis=0)
    im = ax.imshow(mean_img, cmap="YlOrRd", vmin=0, vmax=16)
    ax.set_title(f"Mean Digit {digit}", fontsize=10.5, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
cbar_ax = fig.add_axes([0.92, 0.15, 0.018, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Mean Intensity (0–16)", fontsize=10)
fig.suptitle("Average Pixel Intensity Pattern per Digit Class",
             fontsize=13, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 0.90, 0.95])
fig.savefig(os.path.join(OUTPUT_DIR, "mean_digit_patterns.png"), bbox_inches="tight")
plt.show()

# 4. Pixel Intensity & Active Pixel Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.6), dpi=130)
non_zero_pixels = X[X > 0]
ax1.hist(non_zero_pixels, bins=np.arange(0.5, 17.5, 1), color=ACCENT_ORANGE,
         edgecolor="white", rwidth=0.85)
ax1.set_xticks(range(1, 17))
ax1.set_xlabel("Non-Zero Pixel Intensity (1–16)")
ax1.set_ylabel("Pixel Frequency")
ax1.set_title("Active Pixel Intensity Distribution")

active_counts = (X > 0).sum(axis=1)
ink_per_digit = [active_counts[y == d] for d in range(10)]
ax2.boxplot(ink_per_digit, tick_labels=range(10), patch_artist=True,
            boxprops=dict(facecolor=SOFT_SAND, edgecolor=ACCENT_TEAL),
            medianprops=dict(color=ACCENT_ORANGE, linewidth=2),
            whiskerprops=dict(color=ACCENT_TEAL),
            capprops=dict(color=ACCENT_TEAL),
            flierprops=dict(marker="o", markersize=3, alpha=0.5))
ax2.set_xlabel("Digit Class (0–9)")
ax2.set_ylabel("Active Pixels per Image (out of 64)")
ax2.set_title("Active Pixel Count by Digit Class")

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "pixel_intensity.png"))
plt.show()

## 4. Feature Engineering & Leak-Free Preprocessing Pipeline

In [ ]:
def engineer_features(X: np.ndarray) -> np.ndarray:
    """
    Expand 64 pixel features to 82 features:
      - 64 raw pixel intensities (0-16)
      - 8 row sums
      - 8 column sums
      - 1 total ink (sum of all pixels)
      - 1 active pixel count (pixels > 0)
    """
    X = np.asarray(X, dtype=np.float64)
    imgs = X.reshape(-1, 8, 8)
    row_sums = imgs.sum(axis=2)
    col_sums = imgs.sum(axis=1)
    total_ink = X.sum(axis=1, keepdims=True)
    active_pixels = (X > 0).sum(axis=1, keepdims=True)
    return np.hstack([X, row_sums, col_sums, total_ink, active_pixels])


def build_pipeline(classifier) -> Pipeline:
    return Pipeline([
        ("feature_engineering", FunctionTransformer(engineer_features, validate=False)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("classifier", classifier),
    ])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train_fe = engineer_features(X_train)
pca_check = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(
    StandardScaler().fit_transform(X_train_fe)
)
print(f"Train / Test split   : {len(X_train)} train / {len(X_test)} test (stratified 80/20)")
print(f"Engineered features  : {X_train_fe.shape[1]} features")
print(f"PCA components (95%) : {pca_check.n_components_} dimensions")

## 5. Model Comparison (5-Fold CV) & Held-Out Test Evaluation

In [ ]:
classifiers = {
    "RBF SVM": SVC(C=7.0, kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_features="sqrt", random_state=RANDOM_STATE
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5, weights="distance", metric="euclidean"
    ),
    "Logistic Regression": LogisticRegression(
        C=1.0, max_iter=1000, random_state=RANDOM_STATE
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted_pipelines = {}

for name, clf in classifiers.items():
    pipe = build_pipeline(clf)
    cv_scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1
    )
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    results.append({
        "Model": name,
        "CV Macro F1": cv_scores["test_score"].mean(),
        "CV Std": cv_scores["test_score"].std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Balanced Acc": balanced_accuracy_score(y_test, y_pred),
        "Test Macro Precision": precision_score(y_test, y_pred, average="macro"),
        "Test Macro Recall": recall_score(y_test, y_pred, average="macro"),
        "Test Macro F1": f1_score(y_test, y_pred, average="macro"),
        "Test ROC-AUC (OvR)": roc_auc_score(
            y_test, y_proba, multi_class="ovr", average="macro"
        ),
    })

cmp_df = pd.DataFrame(results).sort_values("CV Macro F1", ascending=False).reset_index(drop=True)
best_model_name = cmp_df.loc[0, "Model"]
best_model = fitted_pipelines[best_model_name]

print(f"Selected Best Model: {best_model_name}\n")
display(cmp_df.round(4))

## 6. Classification Report, Model Comparison Chart & Confusion Matrix

In [ ]:
y_test_pred = best_model.predict(X_test)
print(f"Per-Class Performance on Held-Out Test Set ({best_model_name}):\n")
print(classification_report(y_test, y_test_pred, digits=4))

# 5. Model Comparison Bar Chart
fig, ax = plt.subplots(figsize=(9.5, 5.0), dpi=130)
x = np.arange(len(cmp_df))
width = 0.26

b1 = ax.bar(x - width, cmp_df["CV Macro F1"], width,
            yerr=cmp_df["CV Std"], capsize=3,
            label="5-Fold CV Macro F1", color=ACCENT_TEAL)
b2 = ax.bar(x, cmp_df["Test Accuracy"], width,
            label="Test Accuracy", color=ACCENT_ORANGE)
b3 = ax.bar(x + width, cmp_df["Test Macro F1"], width,
            label="Test Macro F1", color="#4a5568")

for bars in (b1, b2, b3):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.004, f"{h:.3f}",
                ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(cmp_df["Model"], fontweight="bold")
ax.set_ylim(0.90, 1.01)
ax.set_ylabel("Score")
ax.set_title("Classifier Comparison: 5-Fold Cross-Validation vs. Held-Out Test Set")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"))
plt.show()

# 6. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(7.5, 6.5), dpi=130)
im = ax.imshow(cm, cmap="Blues")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Sample Count", fontsize=10)

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > thresh else INK_DARK,
                fontsize=10, fontweight="bold" if i == j else "normal")

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("Predicted Digit")
ax.set_ylabel("True Digit")
ax.set_title(f"Confusion Matrix — {best_model_name} (Test Accuracy: {cmp_df.loc[0, 'Test Accuracy']*100:.2f}%)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"))
plt.show()

## 7. Interactive Inference (Test Sample & Custom Image Upload in Colab)

In [ ]:
def preprocess_external_image(image_path: str) -> np.ndarray:
    """Crop, center, downsample to 8x8, and normalize to [0, 16]."""
    img = Image.open(image_path).convert("L")
    arr = np.asarray(img, dtype=np.float64)
    if arr.mean() > 127:
        arr = 255.0 - arr

    ys, xs = np.where(arr > 40)
    if len(xs) > 0 and len(ys) > 0:
        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()
        size = int(max(x1 - x0, y1 - y0) * 1.25 + 4)
        cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
        half = size // 2
        padded = np.zeros((size, size), dtype=np.float64)
        src_x0, src_y0 = max(0, int(cx - half)), max(0, int(cy - half))
        src_x1, src_y1 = min(arr.shape[1], int(cx - half) + size), min(arr.shape[0], int(cy - half) + size)
        dst_x0, dst_y0 = src_x0 - int(cx - half), src_y0 - int(cy - half)
        padded[dst_y0:dst_y0 + (src_y1 - src_y0), dst_x0:dst_x0 + (src_x1 - src_x0)] = arr[src_y0:src_y1, src_x0:src_x1]
        arr = padded

    resized = Image.fromarray(arr.astype(np.uint8)).resize((8, 8), Image.Resampling.BILINEAR)
    px = np.asarray(resized, dtype=np.float64).flatten()
    max_px = px.max() if px.max() > 0 else 1.0
    return np.round(np.clip((px / max_px) * 16.0, 0, 16)).reshape(1, 64)


def predict_and_visualize(sample_64: np.ndarray, title: str = "Input 8×8 Digit"):
    probs = best_model.predict_proba(sample_64)[0]
    pred_class = int(np.argmax(probs))
    confidence = probs[pred_class] * 100.0

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.5, 3.2), dpi=130,
                                   gridspec_kw={"width_ratios": [1, 2.2]})
    ax1.imshow(sample_64.reshape(8, 8), cmap="gray", vmin=0, vmax=16)
    ax1.set_title(f"{title}\nPred: {pred_class} ({confidence:.1f}%)", fontweight="bold")
    ax1.set_xticks([])
    ax1.set_yticks([])

    colors = [ACCENT_ORANGE if i == pred_class else ACCENT_TEAL for i in range(10)]
    ax2.barh(range(10), probs * 100, color=colors)
    ax2.set_yticks(range(10))
    ax2.invert_yaxis()
    ax2.set_xlim(0, 108)
    ax2.set_xlabel("Probability (%)")
    ax2.set_title("Class Confidence Distribution")
    for i, p in enumerate(probs):
        if p >= 0.01:
            ax2.text(p * 100 + 1.5, i, f"{p*100:.1f}%", va="center", fontsize=8.5)
    fig.tight_layout()
    plt.show()


# Demo prediction on held-out test sample #0
predict_and_visualize(X_test[0:1], title=f"Test Sample (True: {y_test[0]})")

In [ ]:
# Optional: Upload your own handwritten digit image (PNG/JPG) in Google Colab
try:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded.keys():
        sample_64 = preprocess_external_image(filename)
        predict_and_visualize(sample_64, title=filename)
except ImportError:
    print("Run this cell inside Google Colab to upload custom digit images.")